In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import time
import math
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image
import os

# ✅ SET REQUIRED ENVIRONMENT VARIABLE FOR ACT
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.environ['DEVICE'] = str(device)
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = "1"

print(f"🔧 Environment setup complete: DEVICE={device}")

In [ ]:
import os
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def load_hdf5(dataset_path):
    if not os.path.isfile(dataset_path):
        print(f'Dataset does not exist at \n{dataset_path}\n')
        return None, None, None, None

    with h5py.File(dataset_path, 'r') as root:
        qpos = root['/observations/qpos'][()]
        qvel = root['/observations/qvel'][()]
        image_dict = dict()
        for cam_name in root[f'/observations/images/'].keys():
            image_dict[cam_name] = root[f'/observations/images/{cam_name}'][()]
        action = root['/action'][()]

    return qpos, qvel, action, image_dict

def load_all_episodes(data_dir):
    """
    Load all HDF5 files from the specified directory
    Args:
        data_dir: directory containing HDF5 files
    Returns:
        combined qpos, qvel, action arrays and list of image dicts
    """
    print(f"Loading episodes from: {data_dir}")
    
    # Find all HDF5 files
    hdf5_files = [f for f in os.listdir(data_dir) if f.endswith('.hdf5')]
    hdf5_files.sort()  # Sort for consistent ordering
    
    if not hdf5_files:
        print("No HDF5 files found in the directory!")
        return None, None, None, None
    
    print(f"Found {len(hdf5_files)} HDF5 files: {hdf5_files}")
    
    all_qpos = []
    all_qvel = []
    all_action = []
    all_image_dicts = []
    
    episode_info = []
    
    for filename in hdf5_files:
        filepath = os.path.join(data_dir, filename)
        print(f"Loading {filename}...")
        
        qpos, qvel, action, image_dict = load_hdf5(filepath)
        
        if qpos is not None:
            print(f"  - Episode length: {len(qpos)} timesteps")
            print(f"  - QPos shape: {qpos.shape}")
            print(f"  - Action shape: {action.shape}")
            
            all_qpos.append(qpos)
            all_qvel.append(qvel)
            all_action.append(action)
            all_image_dicts.append(image_dict)
            
            episode_info.append({
                'filename': filename,
                'length': len(qpos),
                'start_idx': sum(len(ep) for ep in all_qpos[:-1]),
                'end_idx': sum(len(ep) for ep in all_qpos)
            })
        else:
            print(f"  - Failed to load {filename}")
    
    if not all_qpos:
        print("No valid episodes loaded!")
        return None, None, None, None
    
    # Concatenate all episodes
    combined_qpos = np.concatenate(all_qpos, axis=0)
    combined_qvel = np.concatenate(all_qvel, axis=0)
    combined_action = np.concatenate(all_action, axis=0)
    
    print(f"\nCombined dataset:")
    print(f"Total episodes: {len(all_qpos)}")
    print(f"Total timesteps: {len(combined_qpos)}")
    print(f"Combined QPos shape: {combined_qpos.shape}")
    print(f"Combined Action shape: {combined_action.shape}")
    
    # Print episode breakdown
    print(f"\nEpisode breakdown:")
    for info in episode_info:
        print(f"  {info['filename']}: {info['length']} steps (indices {info['start_idx']}-{info['end_idx']-1})")
    
    return combined_qpos, combined_qvel, combined_action, all_image_dicts

# Load all episodes from the directory
data_dir = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\task1c'
qpos, qvel, action, image_dict_list = load_all_episodes(data_dir)

# ✅ FIXED: Don't call exit(), just check and continue
if qpos is None:
    print("❌ Failed to load any data!")
    print("🔧 Please check your data directory and files")
else:
    print("✅ Data loaded successfully!")
    
    # Create DataFrames with joint headers
    joint_headers = ['timestamp', 'b', 's', 'e', 't', 'r', 'g']
    
    # Create timestamp column (assuming sequential timesteps across all episodes)
    num_timesteps = len(qpos)
    timestamps = np.arange(num_timesteps)
    
    # Create qpos DataFrame
    qpos_data = np.column_stack([timestamps, qpos])
    qpos_df = pd.DataFrame(qpos_data, columns=joint_headers)
    
    # Create action DataFrame
    action_data = np.column_stack([timestamps, action])
    action_df = pd.DataFrame(action_data, columns=joint_headers)
    
    print("\nDataFrame Summary:")
    print("QPos DataFrame shape:", qpos_df.shape)
    print("Action DataFrame shape:", action_df.shape)
    print("\nQPos DataFrame head:")
    print(qpos_df.head())
    print("\nAction DataFrame head:")
    print(action_df.head())
    
    # Check the data ranges for each joint
    print("\nQPos data ranges:")
    for col in joint_headers[1:]:  # Skip timestamp
        print(f"{col}: {qpos_df[col].min():.3f} to {qpos_df[col].max():.3f}")
    
    print("\nAction data ranges:")
    for col in joint_headers[1:]:  # Skip timestamp
        print(f"{col}: {action_df[col].min():.3f} to {action_df[col].max():.3f}")
    
    print("\n✅ Data loading complete! Ready for training.")

In [ ]:
# ========================================
# OFFICIAL ACT IMPORTS AND SETUP
# ========================================

import sys
import os
from copy import deepcopy
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ✅ ADD OFFICIAL ACT PATHS
sys.path.append(r'C:\Users\Administrator\Documents\transformer\ACT-Shaka')
sys.path.append(r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\detr')
sys.path.append(r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\training')

# ✅ IMPORT OFFICIAL ACT COMPONENTS
from training.policy import ACTPolicy
from detr.main import build_ACT_model_and_optimizer

print("✅ Official ACT components imported successfully!")

In [ ]:
# ========================================
# OFFICIAL ACT POLICY CREATION
# ========================================

def create_official_act_policy(state_dim=6, num_queries=100, hidden_dim=512, 
                              camera_names=['front'], kl_weight=100, device=None):
    """
    Create ACTPolicy exactly like the official repository
    Uses the official build_ACT_model_and_optimizer() function
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # ✅ OFFICIAL ARGS OVERRIDE (matches official config.py)
    args_override = {
        'policy_class': 'ACT',
        'kl_weight': kl_weight,
        'chunk_size': num_queries,
        'hidden_dim': hidden_dim,
        'dim_feedforward': 2048,
        'lr': 5e-5,
        'backbone': 'resnet34',
        'loss_function': 'l1',
        'seed': 42,
        'enc_layers': 4,
        'dec_layers': 6,
        'nheads': 8,
        'camera_names': camera_names,
        'state_dim': state_dim,
        'lr_backbone': 1e-5,
        'weight_decay': 1e-4,
        'dropout': 0.1,
        'pre_norm': False,
        'num_queries': num_queries,
    }
    
    # ✅ CREATE OFFICIAL ACT POLICY
    # This internally calls build_ACT_model_and_optimizer() 
    # which creates the unified model (backbone + transformer + heads)
    policy = ACTPolicy(args_override)
    policy = policy.to(device)
    
    # Display model information
    total_params = sum(p.numel() for p in policy.parameters() if p.requires_grad)
    model_params = sum(p.numel() for p in policy.model.parameters() if p.requires_grad)
    
    print(f"✅ Official ACTPolicy Created:")
    print(f"   Total parameters: {total_params/1e6:.2f}M")
    print(f"   Model parameters: {model_params/1e6:.2f}M")
    print(f"   KL weight: {policy.kl_weight}")
    print(f"   Queries: {policy.model.num_queries}")
    print(f"   Backbone: {args_override['backbone']}")
    print(f"   Hidden dim: {args_override['hidden_dim']}")
    print(f"   Compatible with official loading: ✅")
    
    return policy

print("✅ Official ACT Policy Creation Ready!")

In [ ]:
# ========================================
# TRAINING UTILITIES (Official ACT Style)
# ========================================

def set_seed(seed):
    """Set random seeds for reproducibility"""
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

def compute_dict_mean(epoch_dicts):
    """Average loss dictionaries across batches"""
    result = {}
    for key in epoch_dicts[0].keys():
        if key in ['predictions', 'is_pad_pred']:
            continue  # Skip non-scalar values
        result[key] = torch.stack([d[key] for d in epoch_dicts]).mean()
    return result

def detach_dict(d):
    """Detach tensors in dictionary for storage"""
    new_d = {}
    for k, v in d.items():
        if isinstance(v, torch.Tensor):
            new_d[k] = v.detach()
        else:
            new_d[k] = v
    return new_d

class ACTDataset(Dataset):
    """Official ACT Dataset matching train.py"""
    def __init__(self, qpos_df, action_df, image_dict_list, chunk_size=100):
        self.qpos_df = qpos_df
        self.action_df = action_df
        self.image_dict_list = image_dict_list
        self.chunk_size = chunk_size
        
        # Total timesteps across ALL episodes
        self.total_timesteps = len(qpos_df)
        
        # Create flattened image array
        self.all_images = []
        for episode_dict in image_dict_list:
            episode_images = episode_dict['front']  # [timesteps, H, W, 3]
            self.all_images.extend(episode_images)
        
        self.all_images = np.array(self.all_images)  # [total_timesteps, H, W, 3]
        
        print(f"📊 Dataset Statistics:")
        print(f"   Total timesteps: {self.total_timesteps}")
        print(f"   Images shape: {self.all_images.shape}")
        print(f"   Chunk size: {chunk_size}")
        
        # Verify data alignment
        assert len(self.all_images) == self.total_timesteps, \
            f"Image count {len(self.all_images)} != timestep count {self.total_timesteps}"
    
    def __len__(self):
        return self.total_timesteps
    
    def __getitem__(self, idx):
        # Get current qpos
        qpos = torch.FloatTensor(self.qpos_df.iloc[idx, 1:].values)  # Skip timestamp
        
        # Get current image
        if idx < len(self.all_images):
            image = self.all_images[idx]  # [H, W, 3] uint8
            image = torch.FloatTensor(image).permute(2, 0, 1) / 255.0  # [3, H, W]
        else:
            image = torch.zeros(3, 480, 640)
        
        # Get action chunk with padding
        action_chunk = torch.zeros(self.chunk_size, 6)
        is_pad = torch.ones(self.chunk_size, dtype=torch.bool)
        
        end_idx = min(idx + self.chunk_size, self.total_timesteps)
        available_actions = end_idx - idx
        
        if available_actions > 0:
            actions_data = self.action_df.iloc[idx:end_idx, 1:].values
            action_chunk[:available_actions] = torch.FloatTensor(actions_data)
            is_pad[:available_actions] = False
        
        return image, qpos, action_chunk, is_pad

def forward_pass_official_style(data, policy, device):
    """
    Forward pass matching official train.py line 17-21
    This is the EXACT forward pass used in the official repository
    """
    image_data, qpos_data, action_data, is_pad = data
    
    # Move to device (matches train.py line 18)
    image_data = image_data.to(device)
    qpos_data = qpos_data.to(device)
    action_data = action_data.to(device)
    is_pad = is_pad.to(device)
    
    # ✅ OFFICIAL FORWARD PASS (matches train.py line 19)
    # This calls ACTPolicy.__call__() which handles:
    # 1. Image normalization
    # 2. Visual feature extraction (backbone)
    # 3. CVAE encoding (if training)
    # 4. Transformer forward pass
    # 5. Loss computation
    return policy(qpos_data, image_data, action_data, is_pad)

print("✅ Training utilities ready!")

In [ ]:
# ========================================
# OFFICIAL TRAINING FUNCTION
# Matches train.py exactly
# ========================================

def train_act_official(train_dataloader, val_dataloader, checkpoint_dir, task='task1c'):
    """
    Training function matching official train.py exactly
    This is a clean version that uses ONLY official ACT components
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # ✅ CREATE OFFICIAL ACT POLICY
    policy = create_official_act_policy(
        state_dim=6,
        num_queries=100,    # Official POLICY_CONFIG
        hidden_dim=512,     # Official POLICY_CONFIG
        camera_names=['front'],
        kl_weight=100,
        device=device
    )
    
    # ✅ USE OFFICIAL OPTIMIZER (created inside ACTPolicy)
    optimizer = policy.optimizer
    
    # Add scheduler (matches official train.py)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 'min', patience=200, factor=0.5)
    
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    train_history = []
    validation_history = []
    min_val_loss = np.inf
    best_ckpt_info = None
    
    num_epochs = 5000  # Official config
    
    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch}')
        
        # ========================================
        # VALIDATION PHASE (Official train.py structure)
        # ========================================
        with torch.inference_mode():
            policy.eval()
            epoch_dicts = []
            
            for batch_idx, data in enumerate(val_dataloader):
                # ✅ OFFICIAL FORWARD PASS (train.py line 17)
                forward_dict = forward_pass_official_style(data, policy, device)
                epoch_dicts.append(forward_dict)
            
            epoch_summary = compute_dict_mean(epoch_dicts)
            validation_history.append(epoch_summary)
            
            epoch_val_loss = epoch_summary['loss']
            scheduler.step(epoch_val_loss)
            
            if epoch_val_loss < min_val_loss:
                min_val_loss = epoch_val_loss
                best_ckpt_info = (epoch, min_val_loss, deepcopy(policy.state_dict()))
        
        print(f'Val loss:   {epoch_val_loss:.5f}')
        summary_string = ''
        for k, v in epoch_summary.items():
            summary_string += f'{k}: {v.item():.3f} '
        print(summary_string)
        
        # ========================================
        # TRAINING PHASE (Official train.py structure)
        # ========================================
        policy.train()
        optimizer.zero_grad()
        train_epoch_dicts = []
        
        for batch_idx, data in enumerate(train_dataloader):
            # ✅ OFFICIAL FORWARD PASS (train.py line 19)
            forward_dict = forward_pass_official_style(data, policy, device)
            
            # Backward pass (train.py lines 79-83)
            loss = forward_dict['loss']
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
            train_epoch_dicts.append(detach_dict(forward_dict))
        
        # Compute training summary
        epoch_summary = compute_dict_mean(train_epoch_dicts)
        epoch_train_loss = epoch_summary['loss']
        train_history.append(epoch_summary)
        
        print(f'Train loss: {epoch_train_loss:.5f}')
        summary_string = ''
        for k, v in epoch_summary.items():
            summary_string += f'{k}: {v.item():.3f} '
        print(summary_string)
        
        # ========================================
        # ✅ OFFICIAL CHECKPOINT SAVING (train.py line 105)
        # ========================================
        save_freq = 50 if epoch < 500 else 500
        if epoch % save_freq == 0:
            ckpt_path = os.path.join(checkpoint_dir, f"policy_epoch_{epoch}_seed_42.ckpt")
            torch.save(policy.state_dict(), ckpt_path)  # Official style
            
            # Verify checkpoint size
            checkpoint_size_mb = os.path.getsize(ckpt_path) / (1024 * 1024)
            print(f"Checkpoint saved: {ckpt_path}")
            print(f"Checkpoint size: {checkpoint_size_mb:.1f} MB")
    
    # ✅ SAVE FINAL CHECKPOINT (train.py line 107)
    ckpt_path = os.path.join(checkpoint_dir, f'policy_last.ckpt')
    torch.save(policy.state_dict(), ckpt_path)
    
    # ✅ SAVE BEST CHECKPOINT (train.py lines 111-114)
    if best_ckpt_info is not None:
        best_epoch, best_val_loss, best_state_dict = best_ckpt_info
        best_ckpt_path = os.path.join(checkpoint_dir, f"policy_best_epoch_{best_epoch}_val_{best_val_loss:.4f}.ckpt")
        torch.save(best_state_dict, best_ckpt_path)
        
        best_size_mb = os.path.getsize(best_ckpt_path) / (1024 * 1024)
        print(f"✅ Best checkpoint saved: {best_ckpt_path}")
        print(f"Size: {best_size_mb:.1f} MB")
        print(f"Expected: ~350-400 MB (matching official ACT)")
    
    return policy, {'train': train_history, 'val': validation_history}

print("✅ Official training function ready!")

In [ ]:
# ========================================
# TRAINING EXECUTION (Official ACT)
# ========================================

def setup_and_train_act():
    """Complete training setup using ONLY official ACT components"""
    
    # Set seed for reproducibility
    set_seed(42)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Training on device: {device}")
    
    # ========================================
    # STEP 1: Create Dataset
    # ========================================
    print("📊 Creating dataset...")
    chunk_size = 100  # Official ACT configuration
    
    dataset = ACTDataset(
        qpos_df=qpos_df,
        action_df=action_df,
        image_dict_list=image_dict_list,
        chunk_size=chunk_size
    )
    
    print(f"Dataset created with {len(dataset)} samples")
    
    # ========================================
    # STEP 2: Create Train/Val Split
    # ========================================
    train_ratio = 0.8
    train_size = int(len(dataset) * train_ratio)
    val_size = len(dataset) - train_size
    
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
    
    # Create data loaders
    batch_size_train = 32   # Memory-efficient for chunk_size=100
    batch_size_val = 32
    
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size_train, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size_val, shuffle=False)
    
    print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")
    
    # ========================================
    # STEP 3: Setup Checkpoint Directory
    # ========================================
    task = 'task1c'
    checkpoint_dir = os.path.join(r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\checkpoints_nb', task)
    
    print(f"📁 Checkpoint directory: {checkpoint_dir}")
    
    # ========================================
    # STEP 4: Start Training
    # ========================================
    print("🚀 Starting official ACT training...")
    print("📝 CONFIGURATION:")
    print("   - Uses official ACTPolicy class")
    print("   - Uses official build_ACT_model_and_optimizer()")
    print("   - Chunk size: 100 (official)")
    print("   - Hidden dim: 512 (official)")
    print("   - Expected checkpoint size: ~350-400 MB")
    
    trained_policy, history = train_act_official(
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        checkpoint_dir=checkpoint_dir,
        task=task
    )
    
    print("✅ Training completed!")
    return trained_policy, history, checkpoint_dir

# ========================================
# 🚀 START TRAINING
# ========================================

print("🎯 Starting Official ACT Training")
print("✅ Uses ONLY official repository components")
print("✅ Compatible with evaluate_custom.py")
print("✅ Matches train.py exactly")

# Start training
trained_policy, training_history, checkpoint_dir = setup_and_train_act()

print(f"\n🎉 Training Complete!")
print(f"📁 Checkpoints saved to: {checkpoint_dir}")
print(f"📊 Training history: {len(training_history['train'])} epochs")

In [ ]:
# ========================================
# DEBUG: Check what caused SystemExit
# ========================================

print("🔍 Debugging SystemExit...")

# Check if data was loaded properly
try:
    print(f"✅ Data variables exist:")
    print(f"   qpos_df shape: {qpos_df.shape if 'qpos_df' in globals() else 'Not found'}")
    print(f"   action_df shape: {action_df.shape if 'action_df' in globals() else 'Not found'}")
    print(f"   image_dict_list length: {len(image_dict_list) if 'image_dict_list' in globals() else 'Not found'}")
    
    if 'qpos_df' in globals():
        print(f"   QPos columns: {list(qpos_df.columns)}")
        print(f"   QPos sample:\n{qpos_df.head(2)}")
    
    if 'image_dict_list' in globals() and len(image_dict_list) > 0:
        print(f"   First episode cameras: {list(image_dict_list[0].keys())}")
        print(f"   First episode image shape: {image_dict_list[0]['front'].shape}")
        
except Exception as e:
    print(f"❌ Error checking data: {e}")

# Check the data directory
data_dir = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\task1c'
print(f"\n📁 Data directory check:")
print(f"   Path exists: {os.path.exists(data_dir)}")
if os.path.exists(data_dir):
    files = [f for f in os.listdir(data_dir) if f.endswith('.hdf5')]
    print(f"   HDF5 files: {files}")
else:
    print(f"   ❌ Directory not found!")

# Check if the exit() was called
import sys
print(f"\n🔧 Python info:")
print(f"   Python version: {sys.version}")
print(f"   Current working directory: {os.getcwd()}")